# Pilot Experiment: CCI Bias Under Gene-Panel Sparsity

**Goal:** Determine whether spatial imputation inflates cell-cell interaction statistics,
and whether regression calibration can correct it.

## Go/No-Go Criteria
1. CCI inflation > 20% across >= 2 imputation methods
2. Regression calibration removes > 50% of false positives
3. Reliability-error correlation > 0.5

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 150

## Step 0: Load Data

In [ ]:
from src.data_loader import load_xenium_breast_cancer, load_reference, get_lr_pairs

# Update these paths to your data
SPATIAL_PATH = '../data/xenium_breast_cancer.h5ad'
REFERENCE_PATH = '../data/scrna_reference.h5ad'  # optional

adata = load_xenium_breast_cancer(SPATIAL_PATH)
lr_pairs = get_lr_pairs(adata)

# Optional: load scRNA-seq reference
# adata_ref = load_reference(REFERENCE_PATH)

## Step 1: Ground Truth CCI

In [ ]:
from src.cci_inference import compute_cci_scores

# Use n_perms=200 for speed in pilot; increase to 1000 for final
cci_full = compute_cci_scores(adata, lr_pairs, layer='counts', n_perms=200)

n_sig_full = cci_full['significant'].sum()
print(f'Significant CCI pairs (ground truth): {n_sig_full}')
print(f'Total LR pairs tested: {len(cci_full)}')
cci_full.head(10)

## Step 2: Simulate Sparse Panel

In [ ]:
from src.panel_simulation import simulate_sparse_panel

PANEL_SIZE = 300
panel_mask = simulate_sparse_panel(
    adata, m=PANEL_SIZE, lr_pairs=lr_pairs, strategy='mixed'
)

## Step 3: Imputation (4 Method Families)

In [ ]:
methods = {}

# Method 1: GNN-based (no reference needed)
from src.imputation.gnn_runner import run_spatial_gnn
print('Running Spatial GNN...')
methods['SpatialGNN'] = run_spatial_gnn(adata, panel_mask)

# Method 2: Diffusion-based (no reference needed)
from src.imputation.diffusion_runner import run_diffusion
print('Running Diffusion...')
methods['Diffusion'] = run_diffusion(adata, panel_mask)

In [ ]:
# Method 3 & 4: Reference-based (uncomment if you have scRNA-seq reference)

# from src.imputation.tangram_runner import run_tangram
# print('Running Tangram...')
# methods['Tangram'] = run_tangram(adata, adata_ref, panel_mask)

# from src.imputation.gimvi_runner import run_gimvi
# print('Running gimVI...')
# methods['gimVI'] = run_gimvi(adata, adata_ref, panel_mask)

## Step 4: CCI on Imputed Data + Regression Calibration

In [ ]:
from src.regression_calibration import estimate_gene_mse, regression_calibrated_cci

cci_imputed = {}
cci_corrected = {}

for method_name, Lambda_hat in methods.items():
    print(f'\n--- {method_name} ---')
    
    # Create imputed AnnData
    adata_imp = adata.copy()
    adata_imp.layers['imputed'] = Lambda_hat
    
    # Naive CCI
    cci_imputed[method_name] = compute_cci_scores(
        adata_imp, lr_pairs, layer='imputed', n_perms=200
    )
    n_imp = cci_imputed[method_name]['significant'].sum()
    print(f'  Naive significant: {n_imp} (+{n_imp - n_sig_full})')
    
    # Estimate MSE
    mse = estimate_gene_mse(adata, Lambda_hat, panel_mask)
    
    # Calibrated CCI
    cci_corrected[method_name] = regression_calibrated_cci(
        adata, Lambda_hat, mse, lr_pairs, n_perms=200
    )
    n_corr = cci_corrected[method_name]['significant'].sum()
    print(f'  Corrected significant: {n_corr}')
    print(f'  Inflation: {(n_imp - n_sig_full)/n_sig_full*100:.1f}%')

## Step 5: Panel Perturbation Stability

In [ ]:
from src.stability_analysis import panel_perturbation_stability

# Use fewer perturbations for pilot speed
stability_results = {}

def quick_impute(ad, mask):
    return run_spatial_gnn(ad, mask, epochs=100)

for method_name in methods:
    print(f'Stability analysis for {method_name}...')
    stability_results[method_name] = panel_perturbation_stability(
        adata, panel_mask, quick_impute, lr_pairs,
        n_perturbations=20, drop_fraction=0.2
    )

## Step 6: Generate 5 Key Figures

In [ ]:
from src.evaluation import PilotEvaluation

evaluator = PilotEvaluation(
    cci_full, cci_imputed, cci_corrected, stability_results
)

evaluator.figure1_significant_counts()

In [ ]:
evaluator.figure2_false_positive_analysis()

In [ ]:
evaluator.figure3_bias_scatter()

In [ ]:
evaluator.figure4_reliability_vs_error()

In [ ]:
evaluator.figure5_calibration_plot()

## Go / No-Go Decision

In [ ]:
print('GO / NO-GO SUMMARY')
print('=' * 50)

for method_name in methods:
    n_imp = cci_imputed[method_name]['significant'].sum()
    n_corr = cci_corrected[method_name]['significant'].sum()
    inflation = n_imp - n_sig_full
    inflation_rate = inflation / n_sig_full if n_sig_full > 0 else 0
    fp_removed = n_imp - n_corr
    
    print(f'\n{method_name}:')
    print(f'  Ground truth: {n_sig_full}')
    print(f'  Imputed:      {n_imp} (+{inflation}, {inflation_rate*100:.1f}%)')
    print(f'  Corrected:    {n_corr} ({fp_removed} FPs removed)')